# Phnom Penh International University — Online Casino Games EDA
Exploratory Data Analysis of the 1.2M-record online casino games dataset.
Story angle: **Time Matters** — how engagement and outcomes vary by hour/day.

## 1) Imports & Setup

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Project root (notebooks/ -> parent = project root)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.eda.utils import load_csv, summarize_dataframe

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

print("✅ Setup complete")
print("ROOT:", ROOT)

✅ Setup complete
ROOT: /home/tong-bora/source-code/ppiu/source-code/data-analytics


## 2) Load Dataset

In [2]:
csv_path = ROOT / "data" / "raw" / "online_casino_games.csv"

if csv_path.exists():
    # Load 200k sample for fast exploration
    df = load_csv(csv_path, nrows=200_000)
    print("✅ Loaded sample shape:", df.shape)
    display(df.head())
else:
    print("❌ CSV not found at:", csv_path)

✅ Loaded sample shape: (200000, 20)


,casino,game,provider,rtp,volatility,jackpot,country_availability,min_bet,max_win,game_type,game_category,license_jurisdiction,release_year,currency,mobile_compatible,free_spins_feature,bonus_buy_available,max_multiplier,languages,last_updated
0,Guts Casino,Texas Hold'em Bonus,Saucify,98.83,Medium,NaN,AR|BR|CH|CO|CY|DE|EE|ES|IE|IL|KE|MT|NZ|PE|PT|R...,1.00,435957.16,poker,Video Poker,MGA,2021,SEK|USD,True,False,False,356.0,EN|ES|TR|IT|JA|DE|FR|CS,2024-09-20
1,Expekt,Mines,Playtech,98.05,Low,NaN,BG|HR|ID|LT|NO|PH|SE|TR,0.25,1396.15,crash,Crash,Curaçao,2018,NZD|BRL|CAD|SEK|AUD,True,False,False,206.0,EN|SV|TR|ZH|PT|ES|FR|JA,2024-02-03
2,Betinia,Mega Vault,Gamomat,95.43,Very High,NaN,CH|ES|GB|HU|ID|IE|IN|LT|NO|SE|SG|VN,0.10,1180.12,slot,3D Slot,Isle of Man,2014,NZD|AUD|BRL|GBP|CAD,True,True,False,1787.0,EN|DE|FI|HU|PT|ZH|FR|JA,2024-07-09
3,Pinnacle,Casino Hold'em Pro,4ThePlayer,99.50,Medium,NaN,BE|BR|MY|NG|PL,0.05,236507.50,poker,Three Card Poker,MGA,2023,DKK|NZD|SEK,True,False,False,369.0,EN|ES|SV|DA|CS|FR|ZH|FI|NO|PT|HU|KO,2024-06-27
4,Winnerz,Solar Star Xtreme,Hacksaw Gaming,97.18,High,NaN,CL|KE|PT|SG|SI,0.20,1093.45,slot,Jackpot Slot,MGA,2022,NOK|GBP,True,True,False,1970.0,EN|JA|PT|KO|ES,2024-04-24


## 3) Schema, Dtypes & Memory

In [3]:
print(df.info(memory_usage="deep"))

categorical_cols = [c for c in df.columns if df[c].dtype == "object"]
print("\n📋 Categorical columns unique counts:")
display(df[categorical_cols].nunique().sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   casino                200000 non-null  str    
 1   game                  200000 non-null  str    
 2   provider              200000 non-null  str    
 3   rtp                   200000 non-null  float64
 4   volatility            200000 non-null  str    
 5   jackpot               21492 non-null   str    
 6   country_availability  200000 non-null  str    
 7   min_bet               200000 non-null  float64
 8   max_win               200000 non-null  float64
 9   game_type             200000 non-null  str    
 10  game_category         200000 non-null  str    
 11  license_jurisdiction  200000 non-null  str    
 12  release_year          200000 non-null  int64  
 13  currency              200000 non-null  str    
 14  mobile_compatible     200000 non-null  bool   
 15  free_spins_

Series([], dtype: float64)

## 4) Missing Values & Duplicates

In [4]:
summary = summarize_dataframe(df)
print("Rows:", summary["rows"], "| Columns:", summary["columns"])

missing = pd.Series(summary["missing_per_column"]).sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing) > 0:
    print("\n⚠️  Missing values:")
    display(missing)
else:
    print("✅ No missing values found")

Rows: 200000 | Columns: 20

⚠️  Missing values:


jackpot           178508
max_multiplier     35919
dtype: int64

## 5) Summary Statistics

In [5]:
numeric_cols = df.select_dtypes(include=["number"]).columns
display(df[numeric_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

,rtp,min_bet,max_win,release_year,max_multiplier
count,200000.000000,200000.000000,2.000000e+05,200000.000000,164081.000000
mean,96.199573,0.466079,3.348710e+05,2017.530530,6121.097324
std,2.610342,0.809667,6.701273e+05,4.180554,11376.293082
min,85.000000,0.010000,1.000700e+02,2010.000000,10.000000
5%,90.530000,0.010000,7.132580e+02,2010.000000,80.000000
25%,95.000000,0.100000,2.069707e+03,2014.000000,282.000000
50%,96.640000,0.200000,1.537190e+04,2018.000000,741.000000
75%,98.090000,0.500000,3.600289e+05,2021.000000,6069.000000
95%,99.500000,1.000000,1.701104e+06,2024.000000,37032.000000
max,99.500000,5.000000,4.999956e+06,2024.000000,50000.000000


## 6) Story Angle — Time Matters

In [6]:
STORY_ANGLE = "Time Matters — how engagement and outcomes vary by hour/day"
print("📖 Story angle:", STORY_ANGLE)
print("\n📋 All columns available for analysis:")
print(df.columns.tolist())

📖 Story angle: Time Matters — how engagement and outcomes vary by hour/day

📋 All columns available for analysis:
['casino', 'game', 'provider', 'rtp', 'volatility', 'jackpot', 'country_availability', 'min_bet', 'max_win', 'game_type', 'game_category', 'license_jurisdiction', 'release_year', 'currency', 'mobile_compatible', 'free_spins_feature', 'bonus_buy_available', 'max_multiplier', 'languages', 'last_updated']
